In [ ]:
import sys
import os
sys.path.append('..')
import pandas as pd
import yfinance as yf


Correlation Analysis

In [ ]:
sys.path.append(os.path.join(os.getcwd(), '..'))  # Go up one level from notebooks
sys.path.append(os.path.join(os.getcwd(), '../src'))  # Direct to src
from src.news_analyzer import NewsAnalyzer
from src.sentiment_analyzer import SentimentAnalyzer
from src.data_integrator import DataIntegrator

Load and prepare news data

In [ ]:
print("🔗 CORRELATION ANALYSIS: News Sentiment vs Stock Returns")
news_analyzer = NewsAnalyzer('../data/raw_analyst_ratings.csv')
news_analyzer.load_data()

# Fix dates and add sentiment
news_analyzer.df['date'] = pd.to_datetime(news_analyzer.df['date'], format='mixed')
sentiment_analyzer = SentimentAnalyzer()
news_analyzer.df['sentiment'] = sentiment_analyzer.analyze_news_sentiment(news_analyzer.df['headline'])

print(f"📰 Processed {len(news_analyzer.df):,} news articles with sentiment")

Merge and analyze correlation

In [ ]:
def get_aligned_stock_data(news_dates: pd.Series, symbol: str = "AAPL") -> pd.DataFrame:
    """Get stock data aligned with news date range."""
    start_date = news_dates.min().strftime('%Y-%m-%d')
    end_date = news_dates.max().strftime('%Y-%m-%d')
    
    stock_data = yf.download(symbol, start=start_date, end=end_date)
    print(f"📈 Downloaded {symbol} data: {start_date} to {end_date} ({len(stock_data)} trading days)")
    return stock_data

price_data = get_aligned_stock_data(news_analyzer.df['date'])


In [ ]:
integrator = DataIntegrator()
merged_data = integrator.align_news_with_prices(news_analyzer.df, price_data)

# Statistical analysis
stats_report = integrator.calculate_correlation_stats()

print("\n📊 STATISTICAL RESULTS:")
print(f"• Pearson Correlation: {stats_report['pearson_correlation']:.3f}")
print(f"• P-value: {stats_report['pearson_p_value']:.3f}")
print(f"• Statistically Significant: {stats_report['significant_pearson']}")
print(f"• Days Analyzed: {stats_report['days_analyzed']}")
print(f"• Average News per Day: {stats_report['avg_news_per_day']:.1f}")

 Create correlation plots and Trading strategy backtest

In [ ]:
integrator.create_correlation_plots(save_path='../results/correlation_analysis.png')

In [ ]:
def simple_sentiment_strategy(merged_data: pd.DataFrame) -> dict:
    """Simple strategy based on sentiment signals."""
    # Buy when sentiment > 0.1, sell when < -0.1
    merged_data = merged_data.copy()
    merged_data['signal'] = 0
    merged_data.loc[merged_data['avg_sentiment'] > 0.1, 'signal'] = 1  # Buy
    merged_data.loc[merged_data['avg_sentiment'] < -0.1, 'signal'] = -1  # Sell
    
    # Calculate strategy returns
    merged_data['strategy_return'] = merged_data['signal'] * merged_data['daily_return'].shift(-1)
    
    # Compare with buy-and-hold
    buy_hold_return = merged_data['daily_return'].sum()
    strategy_total_return = merged_data['strategy_return'].sum()
    
    return {
        'buy_hold_return': buy_hold_return,
        'strategy_return': strategy_total_return,
        'outperformance': strategy_total_return - buy_hold_return,
        'signal_days': (merged_data['signal'] != 0).sum(),
        'win_rate': (merged_data[merged_data['signal'] != 0]['strategy_return'] > 0).mean()
    }

strategy_results = simple_sentiment_strategy(merged_data)
print("\n🎯 TRADING STRATEGY RESULTS:")
print(f"• Buy & Hold Return: {strategy_results['buy_hold_return']:.3f}")
print(f"• Sentiment Strategy Return: {strategy_results['strategy_return']:.3f}")
print(f"• Outperformance: {strategy_results['outperformance']:.3f}")
print(f"• Win Rate: {strategy_results['win_rate']:.1%}")

print("\n✅ CORRELATION ANALYSIS COMPLETE")